In [1]:
import sqlite3
import pandas as pd

In [2]:
df = pd.read_csv('train.csv')
conn = sqlite3.connect(':memory:')
df.to_sql('titanic', conn, index=False)

pd.read_sql('SELECT * FROM titanic LIMIT 5', conn)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
pd.read_sql('''
SELECT Sex, AVG(Survived) AS survival_rate
FROM titanic
GROUP BY Sex
''', conn)

,Sex,survival_rate
0,female,0.742038
1,male,0.188908


In [4]:
pd.read_sql('''
SELECT Pclass, AVG(Survived) AS survival_rate
FROM titanic
GROUP BY Pclass
''', conn)

,Pclass,survival_rate
0,1,0.629630
1,2,0.472826
2,3,0.242363


In [5]:
pd.read_sql('''
SELECT Pclass, AVG(Survived) AS survival_rate
FROM titanic
WHERE Sex = 'female'
GROUP BY Pclass
''', conn)

,Pclass,survival_rate
0,1,0.968085
1,2,0.921053
2,3,0.500000


In [6]:
pd.read_sql('''
SELECT Pclass, AVG(Survived) AS survival_rate
FROM titanic
WHERE Sex = 'male'
GROUP BY Pclass
''', conn)

,Pclass,survival_rate
0,1,0.368852
1,2,0.157407
2,3,0.135447


In [7]:
pd.read_sql('''
SELECT Pclass, COUNT(*) AS passengers, AVG(Survived) AS survival_rate
FROM titanic
GROUP BY Pclass
''', conn)

,Pclass,passengers,survival_rate
0,1,216,0.629630
1,2,184,0.472826
2,3,491,0.242363


In [8]:
pd.read_sql('''
SELECT Pclass, COUNT(*) AS passengers, AVG(Survived) AS survival_rate
FROM titanic
GROUP BY Pclass
ORDER BY survival_rate DESC
''', conn)

,Pclass,passengers,survival_rate
0,1,216,0.629630
1,2,184,0.472826
2,3,491,0.242363


In [9]:
pd.read_sql('''
SELECT Pclass, COUNT(*) AS passengers, AVG(Survived) AS survival_rate
FROM titanic
GROUP BY Pclass
ORDER BY survival_rate ASC
''', conn)

,Pclass,passengers,survival_rate
0,3,491,0.242363
1,2,184,0.472826
2,1,216,0.629630


In [10]:
pd.read_sql('''
SELECT Embarked, COUNT(*) AS passengers, AVG(Survived) AS survival_rate
FROM titanic
WHERE Embarked IS NOT NULL
GROUP BY Embarked
ORDER BY survival_rate DESC
''', conn)

,Embarked,passengers,survival_rate
0,C,168,0.553571
1,Q,77,0.389610
2,S,644,0.336957


In [11]:
pd.read_sql('''
SELECT Embarked, Pclass, COUNT(*) AS passengers
FROM titanic
WHERE Embarked IS NOT NULL
GROUP BY Embarked, Pclass
ORDER BY Embarked, Pclass
''', conn)

,Embarked,Pclass,passengers
0,C,1,85
1,C,2,17
2,C,3,66
3,Q,1,2
4,Q,2,3
5,Q,3,72
6,S,1,127
7,S,2,164
8,S,3,353


Result: Cherbourg had the highest survival rate, and about 51% of its passengers were in Class 1 (vs. 20% in Southampton). However, this is not conclusive: Queenstown, which was almost entirely Class 3, still had a higher survival rate than Southampton. So ticket class alone does not explain the difference.

In [12]:
ports = pd.DataFrame({
    'Code': ['S', 'C', 'Q'],
    'PortName': ['Southampton', 'Cherbourg', 'Queenstown']
})
ports.to_sql('ports', conn, index=False)
pd.read_sql('SELECT * FROM ports', conn)

,Code,PortName
0,S,Southampton
1,C,Cherbourg
2,Q,Queenstown


In [13]:
pd.read_sql('''
SELECT titanic.Name, titanic.Embarked, ports.PortName
FROM titanic
JOIN ports ON titanic.Embarked = ports.Code
LIMIT 5
''', conn)

,Name,Embarked,PortName
0,"Braund, Mr. Owen Harris",S,Southampton
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",C,Cherbourg
2,"Heikkinen, Miss. Laina",S,Southampton
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",S,Southampton
4,"Allen, Mr. William Henry",S,Southampton


In [14]:
pd.read_sql('''
SELECT ports.PortName, COUNT(*) AS passengers, AVG(titanic.Survived) AS survival_rate
FROM titanic
JOIN ports ON titanic.Embarked = ports.Code
GROUP BY ports.PortName
ORDER BY survival_rate DESC
''', conn)

,PortName,passengers,survival_rate
0,Cherbourg,168,0.553571
1,Queenstown,77,0.389610
2,Southampton,644,0.336957
